# F-09 Whisper 모델 비교 평가

| 모델 | 설명 |
|------|------|
| baseline | 순정 whisper-large-v3-turbo (파인튜닝 없음) |
| final1 | 1차 튜닝 (suppress_tokens 버그 있음) |
| final2 | 2차 튜닝 — **확정 모델** |
| final3 | 3차 튜닝 (SP 태그 오염 데이터) |

**실행 순서**: 셀 01 → 02 → 03

In [ ]:
# 셀 01 — 라이브러리 설치 + Drive 마운트 + 경로 설정
!pip install -q transformers datasets peft accelerate evaluate jiwer librosa soundfile
!pip install -q "torchao>=0.16.0"

from google.colab import drive
from pathlib import Path
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT     = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH   = DRIVE_ROOT / 'processed/senior_speech_v2'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/whisper-senior'

MODEL_PATHS = {
    'baseline': None,                          # 순정 모델 (LoRA 없음)
    'final1'  : CHECKPOINT_DIR / 'final',      # 1차 튜닝
    'final2'  : CHECKPOINT_DIR / 'final2',     # 2차 튜닝 — 확정 모델
    'final3'  : CHECKPOINT_DIR / 'final3',     # 3차 튜닝 (오염 데이터)
}

for name, path in MODEL_PATHS.items():
    if path is not None:
        print(f'{name}: {path}  존재={path.exists()}')
    else:
        print(f'{name}: 순정 모델')

print(f'데이터셋: {DATASET_PATH}  존재={DATASET_PATH.exists()}')


In [ ]:
# 셀 02 — 평가 함수 정의
import re
import torch
import evaluate
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any
from datasets import load_from_disk
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

MODEL_ID      = 'openai/whisper-large-v3-turbo'
EVAL_SAMPLES  = 500
BATCH_SIZE    = 8
NUM_BEAMS     = 1
PUNCT_PATTERN = re.compile(r'[.?!,。、]')
cer_metric    = evaluate.load('cer')

def clean_text(text: str) -> str:
    return PUNCT_PATTERN.sub('', text).replace(' ', '').strip()

@dataclass
class EvalCollator:
    processor: Any
    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        return {'input_features': inputs.input_features, 'texts': texts}

def evaluate_model(name, adapter_path, dataset):
    print(f'
[{name}] 로드 중...')
    processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')
    base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    base_model.generation_config.language           = 'korean'
    base_model.generation_config.task               = 'transcribe'
    base_model.generation_config.forced_decoder_ids = None

    if adapter_path is not None:
        # LoRA 어댑터 로드
        model = PeftModel.from_pretrained(base_model, str(adapter_path), is_trainable=False)
    else:
        model = base_model

    model = model.half().to('cuda').eval()

    eval_subset = dataset['validation'].select(range(EVAL_SAMPLES))
    loader      = DataLoader(eval_subset, batch_size=BATCH_SIZE, collate_fn=EvalCollator(processor))
    gen_kwargs  = dict(language='korean', task='transcribe', num_beams=NUM_BEAMS)

    all_preds, all_refs = [], []
    for batch in loader:
        input_feats = batch['input_features'].to('cuda', dtype=torch.float16)
        with torch.no_grad():
            pred_ids = model.generate(input_features=input_feats, **gen_kwargs)
        preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
        all_preds.extend([clean_text(p) for p in preds])
        all_refs.extend( [clean_text(r) for r in batch['texts']])

    cer = cer_metric.compute(predictions=all_preds, references=all_refs)
    print(f'[{name}] CER: {cer:.4f}  ({cer * 100:.2f}%)')

    # 샘플 5개 출력
    print(f'  예측 vs 정답 샘플 3개:')
    for i in range(3):
        print(f'    정답: {all_refs[i]}')
        print(f'    예측: {all_preds[i]}')

    # VRAM 해제
    del model, base_model
    torch.cuda.empty_cache()

    return cer

dataset = load_from_disk(str(DATASET_PATH))
print('데이터셋 로드 완료')
print(dataset)


In [ ]:
# 셀 03 — 4개 모델 순차 평가
# 모델마다 로드→평가→VRAM 해제 반복 (A100 40GB 기준 안전)
results = {}

for name, path in MODEL_PATHS.items():
    cer = evaluate_model(name, path, dataset)
    results[name] = round(cer * 100, 2)

# 결과 요약
print('
' + '=' * 45)
print('모델 비교 결과 요약')
print('=' * 45)
print(f'{'모델':<12} {'CER':>8}  {'비고'}')
print('-' * 45)
notes = {
    'baseline': '순정 모델',
    'final1'  : '1차 튜닝 (suppress_tokens 버그)',
    'final2'  : '2차 튜닝 — 확정 모델',
    'final3'  : '3차 튜닝 (SP 태그 오염 데이터)',
}
for name, cer in results.items():
    print(f'{name:<12} {cer:>7.2f}%  {notes[name]}')
print('=' * 45)


In [ ]:
# 셀 04 — 결과 요약 표 출력 (노션 복사용)

notes = {
    'baseline': '순정 모델 (파인튜닝 없음)',
    'final1'  : '1차 튜닝 (suppress_tokens 버그)',
    'final2'  : '2차 튜닝 — 확정 모델',
    'final3'  : '3차 튜닝 (SP 태그 오염 데이터)',
}

baseline_cer = results.get('baseline', None)

col1, col2, col3, col4 = 12, 10, 16, 30

header = f"{'모델':<{col1}} {'CER (%)':>{col2}}  {'baseline 대비':>{col3}}  {'비고'}"
sep    = '-' * (col1 + col2 + col3 + col4 + 6)

print(sep)
print(header)
print(sep)

for name in ['baseline', 'final1', 'final2', 'final3']:
    if name not in results:
        print(f'{name:<{col1}} {"측정 안됨":>{col2}}')
        continue
    cer = results[name]
    diff_str = '-' if (baseline_cer is None or name == 'baseline') else f'{cer - baseline_cer:+.2f}%p'
    print(f'{name:<{col1}} {cer:>{col2}.2f}%  {diff_str:>{col3}}  {notes[name]}')

print(sep)
best = min(results, key=results.get)
print(f'  최저 CER: {best}  ({results[best]:.2f}%)')
print(sep)
